In [ ]:
import json
import re
import sys
import warnings
from pathlib import Path

import pandas as pd

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from speech_recognition.config import (
    ALLOWED_EVALUATION_STRATEGIES,
    ALLOWED_FEATURE_PIPELINES,
    ALLOWED_MODEL_FAMILIES,
    ALLOWED_SCHEDULERS,
)

warnings.filterwarnings("ignore")

# Data load

In [2]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


project_root = Path.cwd().resolve().parent
mlruns_dir = project_root / "mlruns" / "1"
run_dirs = sorted(path for path in mlruns_dir.iterdir() if path.is_dir())

per_class_rows = []
eval_rows = []

for run_dir in run_dirs:
    run_id = run_dir.name
    artifact_dir = run_dir / "artifacts"
    payload_path = artifact_dir / "run_payload.json"

    # eval data: payload metrics
    if payload_path.exists():
        payload = load_json(payload_path)
        metrics = payload.get("metrics", {})
        checkpoint_path = payload.get("checkpoint_path", "")

        eval_rows.append(
            {
                "run_id": run_id,
                "phase": payload.get("phase"),
                "metric_phase": payload.get("metric_phase"),
                "epoch": payload.get("epoch"),
                "step": payload.get("step"),
                "checkpoint_path": checkpoint_path,
                "macro_f1": metrics.get("macro_f1"),
                "command_macro_f1": metrics.get("command_macro_f1"),
                "core_command_macro_f1": metrics.get("core_command_macro_f1"),
                "unknown_f1": metrics.get("unknown_f1"),
                "silence_f1": metrics.get("silence_f1"),
                "macro_f1_nc": metrics.get("macro_f1_nc"),
                "unknown_to_command_leakage": metrics.get("unknown_to_command_leakage"),
                "silence_false_trigger_rate": metrics.get("silence_false_trigger_rate"),
            }
        )

        # per-class eval data
        for class_name, class_scores in metrics.get("per_class", {}).items():
            per_class_rows.append(
                {
                    "run_id": run_id,
                    "phase": payload.get("phase"),
                    "class_name": class_name,
                    "precision": class_scores.get("precision"),
                    "recall": class_scores.get("recall"),
                    "f1": class_scores.get("f1"),
                }
            )


df_eval = pd.DataFrame(eval_rows)
df_per_class = pd.DataFrame(per_class_rows)

if not df_eval.empty:
    df_eval = df_eval.sort_values("run_id").reset_index(drop=True)
if not df_per_class.empty:
    df_per_class = df_per_class.sort_values(["run_id", "class_name"]).reset_index(drop=True)


print(f"\nEval metrics ({len(df_eval)} runs):")
display(df_eval.head())
print(f"\nPer-class metrics ({len(df_per_class)} rows):")
display(df_per_class.head(12))


Eval metrics (136 runs):


,run_id,phase,metric_phase,epoch,step,checkpoint_path,macro_f1,command_macro_f1,core_command_macro_f1,unknown_f1,silence_f1,macro_f1_nc,unknown_to_command_leakage,silence_false_trigger_rate
0,01efede02143439c83433645c1e7a778,phase_1,val,55,17215,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.540173,0.648208,0.648208,0.0,0.0,0.0,0.0,0.0
1,033d62c906e540649556d75c60108531,phase_3,val,26,8138,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.669277,0.803132,0.803132,0.0,0.0,0.0,0.0,0.0
2,0b202ae436594c23b2fe330844f70eb3,phase_3,val,11,3443,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.015152,0.018182,0.018182,0.0,0.0,0.0,0.0,0.0
3,113b2e8c8a6e404890dea59f78c58804,None,val,13,15535,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.0,0.0,0.0,1.0,1.0
4,115f6389b4e543c0b8cd6cf0bfe3f173,phase_3,val,53,16589,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.697054,0.836465,0.836465,0.0,0.0,0.0,0.0,0.0



Per-class metrics (1632 rows):


,run_id,phase,class_name,precision,recall,f1
0,01efede02143439c83433645c1e7a778,phase_1,__silence__,0.000000,0.000,0.000000
1,01efede02143439c83433645c1e7a778,phase_1,__unknown__,0.000000,0.000,0.000000
2,01efede02143439c83433645c1e7a778,phase_1,down,0.552632,0.504,0.527197
3,01efede02143439c83433645c1e7a778,phase_1,go,0.527778,0.456,0.489270
4,01efede02143439c83433645c1e7a778,phase_1,left,0.655797,0.724,0.688213
5,01efede02143439c83433645c1e7a778,phase_1,no,0.683594,0.700,0.691700
6,01efede02143439c83433645c1e7a778,phase_1,off,0.593548,0.368,0.454321
7,01efede02143439c83433645c1e7a778,phase_1,on,0.479109,0.688,0.564860
8,01efede02143439c83433645c1e7a778,phase_1,right,0.660000,0.660,0.660000
9,01efede02143439c83433645c1e7a778,phase_1,stop,0.937500,0.840,0.886076


# Fill phase column

In [3]:
def fill_phase(row):
    if "phase" in row and pd.notna(row["phase"]):
        return row["phase"]

    match = re.search(r"phase_(\d+)", row["checkpoint_path"])
    if match:
        return f"phase_{match.group(1)}"
    return None


df_eval["phase"] = df_eval.apply(
    lambda row: fill_phase(row),
    axis=1,
)

df_per_class = df_per_class.drop(columns=["phase"]).merge(
    df_eval[["run_id", "phase"]],
    on="run_id",
    how="left",
)

df_eval.phase.isna().sum(), df_per_class.phase.isna().sum()

(np.int64(0), np.int64(0))

# Config extraction

In [4]:
def extract_trial_config(checkpoint_path: str) -> str:
    """Extract trial config name from checkpoint path."""
    if not checkpoint_path or pd.isna(checkpoint_path):
        return None
    parts = checkpoint_path.split("/runs/")
    if len(parts) == 2:
        remainder = parts[1]
        return remainder.split("/")[0]
    return None


def extract_trial_seed(trial_config: str) -> int:
    """Extract seed from trial config."""
    if not trial_config or pd.isna(trial_config):
        return None
    match = re.search(r"_seed_(\d+)$", trial_config)
    return int(match.group(1)) if match else None


def extract_config_middle(trial_config: str) -> str:
    """Extract config string (everything between trial_XX_ and _seed_)."""
    if not trial_config or pd.isna(trial_config):
        return None
    match = re.search(r"^trial_\d+_(.*)_seed_\d+$", trial_config)
    return match.group(1) if match else None


df_eval["trial_config"] = df_eval["checkpoint_path"].apply(extract_trial_config)
df_eval["trial_seed"] = df_eval["trial_config"].apply(extract_trial_seed)
df_eval["config_str"] = df_eval["trial_config"].apply(extract_config_middle)

print(f"Extracted trial configs: {df_eval['trial_config'].notna().sum()} / {len(df_eval)}")
display(df_eval[["run_id", "trial_config", "trial_seed", "config_str"]].head(15))

Extracted trial configs: 136 / 136


,run_id,trial_config,trial_seed,config_str
0,01efede02143439c83433645c1e7a778,trial_30_mel_specaugment_xlstm_seed_2003,2003,mel_specaugment_xlstm
1,033d62c906e540649556d75c60108531,trial_13_ast_linear_interp_dropout_0.5_seed_42,42,ast_linear_interp_dropout_0.5
2,0b202ae436594c23b2fe330844f70eb3,trial_07_ast_mlp_256_interp_dropout_0.5_seed_0,0,ast_mlp_256_interp_dropout_0.5
3,113b2e8c8a6e404890dea59f78c58804,trial_01_two_stage_detector_trial_01_ast_linea...,0,two_stage_detector_trial_01_ast_linear_interp_...
4,115f6389b4e543c0b8cd6cf0bfe3f173,trial_19_ast_mlp_256_interp_dropout_0.1_seed_2003,2003,ast_mlp_256_interp_dropout_0.1
5,11b114a41d564f359864dff904b4eaa2,trial_05_mlp_mixer_dropout_0.0_head_l2_True_se...,42,mlp_mixer_dropout_0.0_head_l2_True
6,149eaa36d2a84307a6842b61aa6819ff,trial_11_high_temporal_mel_xlstm_seed_42,42,high_temporal_mel_xlstm
7,19915d2065c84ec385cdc2a18e6e3744,trial_04_convnext_sd_0.2_kernel_7_seed_42,42,convnext_sd_0.2_kernel_7
8,1e7c001bf663478fbd092a6b3dcb4f13,trial_08_two_stage_detector_trial_01_mlp_mixer...,42,two_stage_detector_trial_01_mlp_mixer_dropout_...
9,2262c543024b4282ba8833a582d0ca2c,trial_04_sampling_control_trial_01_convnext_sd...,0,sampling_control_trial_01_convnext_sd_0.0_kern...


In [5]:
df1 = df_eval[df_eval["phase"] == "phase_1"]
df2 = df_eval[df_eval["phase"] == "phase_2"]
df3 = df_eval[df_eval["phase"] == "phase_3"]
df4 = df_eval[df_eval["phase"] == "phase_4"]

## Phase 1

In [6]:
df1["config_str"].unique()

array(['mel_specaugment_xlstm', 'high_temporal_mel_xlstm',
       'mel_specaugment_convnext', 'mfcc_convnext',
       'mel_spectrogram_xlstm', 'mel_spectrogram_convnext',
       'pcen_convnext', 'mfcc_xlstm', 'pcen_xlstm',
       'high_temporal_mel_convnext'], dtype=object)

In [7]:
df1["feature_pipeline"] = df1["config_str"].apply(
    lambda s: next((feat for feat in ALLOWED_FEATURE_PIPELINES if feat in s), None)
)
df1["model_family"] = df1["config_str"].apply(
    lambda s: next((family for family in ALLOWED_MODEL_FAMILIES if family in s), None)
)

In [8]:
df1

,run_id,phase,metric_phase,epoch,step,checkpoint_path,macro_f1,command_macro_f1,core_command_macro_f1,unknown_f1,silence_f1,macro_f1_nc,unknown_to_command_leakage,silence_false_trigger_rate,trial_config,trial_seed,config_str,feature_pipeline,model_family
0,01efede02143439c83433645c1e7a778,phase_1,val,55,17215,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.540173,0.648208,0.648208,0.0,0.0,0.0,0.0,0.0,trial_30_mel_specaugment_xlstm_seed_2003,2003,mel_specaugment_xlstm,mel_specaugment,xlstm
6,149eaa36d2a84307a6842b61aa6819ff,phase_1,val,56,17528,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.330772,0.396927,0.396927,0.0,0.0,0.0,0.0,0.0,trial_11_high_temporal_mel_xlstm_seed_42,42,high_temporal_mel_xlstm,high_temporal_mel,xlstm
17,3466252bf2fa4efc98095ac1322965d1,phase_1,val,41,12833,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.320741,0.384889,0.384889,0.0,0.0,0.0,0.0,0.0,trial_12_high_temporal_mel_xlstm_seed_2003,2003,high_temporal_mel_xlstm,high_temporal_mel,xlstm
25,3b75b67d724e4212a29e17382cd38a60,phase_1,val,11,3443,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.015152,0.018182,0.018182,0.0,0.0,0.0,0.0,0.0,trial_27_mel_specaugment_convnext_seed_2003,2003,mel_specaugment_convnext,mel_specaugment,convnext
32,42852d78afab48d5a52a795e9067b4dd,phase_1,val,31,9703,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.703280,0.843936,0.843936,0.0,0.0,0.0,0.0,0.0,trial_13_mfcc_convnext_seed_0,0,mfcc_convnext,mfcc,convnext
36,52e2061ed18448a6816fd198038c6393,phase_1,val,38,11894,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.544402,0.653282,0.653282,0.0,0.0,0.0,0.0,0.0,trial_04_mel_spectrogram_xlstm_seed_0,0,mel_spectrogram_xlstm,mel_spectrogram,xlstm
44,6075167b17294671862e1a61a5c9a645,phase_1,val,45,14085,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.540905,0.649086,0.649086,0.0,0.0,0.0,0.0,0.0,trial_05_mel_spectrogram_xlstm_seed_42,42,mel_spectrogram_xlstm,mel_spectrogram,xlstm
55,6e9114cac7114cbdb933a64ccca94e92,phase_1,val,11,3443,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.015152,0.018182,0.018182,0.0,0.0,0.0,0.0,0.0,trial_26_mel_specaugment_convnext_seed_42,42,mel_specaugment_convnext,mel_specaugment,convnext
57,7146b327267b437c93c3a36a58260666,phase_1,val,51,15963,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.740798,0.888958,0.888958,0.0,0.0,0.0,0.0,0.0,trial_03_mel_spectrogram_convnext_seed_2003,2003,mel_spectrogram_convnext,mel_spectrogram,convnext
61,738b4d7ea6cd4d3182d604153a184768,phase_1,val,43,13459,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.343172,0.411806,0.411806,0.0,0.0,0.0,0.0,0.0,trial_10_high_temporal_mel_xlstm_seed_0,0,high_temporal_mel_xlstm,high_temporal_mel,xlstm


## Phase 2

In [9]:
df2.config_str.unique()

array(['mel_spectrogram_xlstm_reduce_on_plateau_wd_0.1',
       'mel_spectrogram_xlstm_reduce_on_plateau_wd_0.01',
       'mel_spectrogram_convnext_reduce_on_plateau_wd_0.01',
       'mel_spectrogram_xlstm_cosine_annealing_warmup_wd_0.01',
       'mel_spectrogram_xlstm_cosine_annealing_warmup_wd_0.1',
       'mel_spectrogram_convnext_cosine_annealing_warmup_wd_0.1',
       'mel_spectrogram_convnext_cosine_annealing_warmup_wd_0.01',
       'mel_spectrogram_convnext_reduce_on_plateau_wd_0.1'], dtype=object)

In [10]:
df2["feature_pipeline"] = "mel_spectrogram"
df2["model_family"] = df2["config_str"].apply(
    lambda s: next((family for family in ALLOWED_MODEL_FAMILIES if f"_{family}_" in s), None)
)

df2["weight_decay"] = df2["config_str"].apply(
    lambda s: ".".join(re.search(r"wd_(\d+).(\d+)", s).groups())
)

df2["scheduler"] = df2["config_str"].apply(
    lambda s: next((scheduler for scheduler in ALLOWED_SCHEDULERS if f"_{scheduler}_" in s), None)
)

## Phase 3

In [11]:
df3.config_str.unique()

array(['ast_linear_interp_dropout_0.5', 'ast_mlp_256_interp_dropout_0.5',
       'ast_mlp_256_interp_dropout_0.1',
       'mlp_mixer_dropout_0.0_head_l2_True', 'convnext_sd_0.2_kernel_7',
       'ast_linear_interp_dropout_0.1',
       'mlp_mixer_dropout_0.2_head_l2_False',
       'ast_mlp_256_learned_dropout_0.1',
       'ast_linear_learned_dropout_0.5', 'ast_linear_learned_dropout_0.1',
       'mlp_mixer_dropout_0.2_head_l2_True',
       'mlp_mixer_dropout_0.0_head_l2_False',
       'ast_mlp_256_learned_dropout_0.5', 'convnext_sd_0.0_kernel_7'],
      dtype=object)

In [12]:
AST_HEAD = ["linear", "mlp_256"]
AST_EMB = ["interp", "learned"]

In [13]:
df3["model_family"] = df3["config_str"].apply(
    lambda s: next((family for family in ALLOWED_MODEL_FAMILIES if f"{family}" in s), None)
)

df3["ast_head"] = df3["config_str"].apply(
    lambda s: next((head for head in AST_HEAD if f"{head}" in s), None)
)

df3["ast_embedding"] = df3["config_str"].apply(
    lambda s: next((emb for emb in AST_EMB if f"{emb}" in s), None)
)

df3["dropout"] = df3["config_str"].apply(
    lambda s: (".".join(re.search(r"dropout_(\d+).(\d+)", s).groups()) if "dropout_" in s else None)
)

df3["kernel_size"] = df3["config_str"].apply(
    lambda s: re.search(r"kernel_(\d+)", s).group(1) if "kernel_" in s else None
)

df3["sd"] = df3["config_str"].apply(
    lambda s: ".".join(re.search(r"sd_(\d+).(\d+)", s).groups()) if "sd_" in s else None
)

df3["head_l2"] = df3["config_str"].apply(
    lambda s: True if "head_l2_True" in s else False if "head_l2_False" in s else None
)

In [14]:
df3

,run_id,phase,metric_phase,epoch,step,checkpoint_path,macro_f1,command_macro_f1,core_command_macro_f1,unknown_f1,...,trial_config,trial_seed,config_str,model_family,ast_head,ast_embedding,dropout,kernel_size,sd,head_l2
1,033d62c906e540649556d75c60108531,phase_3,val,26,8138,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.669277,0.803132,0.803132,0.0,...,trial_13_ast_linear_interp_dropout_0.5_seed_42,42,ast_linear_interp_dropout_0.5,ast,linear,interp,0.5,None,None,None
2,0b202ae436594c23b2fe330844f70eb3,phase_3,val,11,3443,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.015152,0.018182,0.018182,0.0,...,trial_07_ast_mlp_256_interp_dropout_0.5_seed_0,0,ast_mlp_256_interp_dropout_0.5,ast,mlp_256,interp,0.5,None,None,None
4,115f6389b4e543c0b8cd6cf0bfe3f173,phase_3,val,53,16589,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.697054,0.836465,0.836465,0.0,...,trial_19_ast_mlp_256_interp_dropout_0.1_seed_2003,2003,ast_mlp_256_interp_dropout_0.1,ast,mlp_256,interp,0.1,None,None,None
5,11b114a41d564f359864dff904b4eaa2,phase_3,val,17,5321,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.050761,0.060913,0.060913,0.0,...,trial_05_mlp_mixer_dropout_0.0_head_l2_True_se...,42,mlp_mixer_dropout_0.0_head_l2_True,mlp_mixer,None,None,0.0,None,None,True
7,19915d2065c84ec385cdc2a18e6e3744,phase_3,val,67,20971,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.746239,0.895487,0.895487,0.0,...,trial_04_convnext_sd_0.2_kernel_7_seed_42,42,convnext_sd_0.2_kernel_7,convnext,None,None,None,7,0.2,None
11,2ab50f05c7b94b57bdadc022859cdab5,phase_3,val,39,12207,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.680569,0.816683,0.816683,0.0,...,trial_17_ast_linear_interp_dropout_0.1_seed_2003,2003,ast_linear_interp_dropout_0.1,ast,linear,interp,0.1,None,None,None
15,2e2c821c93e34b00b56b686ce5c603fd,phase_3,val,68,21284,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.707229,0.848674,0.848674,0.0,...,trial_05_ast_linear_interp_dropout_0.5_seed_0,0,ast_linear_interp_dropout_0.5,ast,linear,interp,0.5,None,None,None
19,380427c82674437f81f6f84af04d5a05,phase_3,val,28,8764,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.722047,0.866456,0.866456,0.0,...,trial_08_mlp_mixer_dropout_0.2_head_l2_False_s...,42,mlp_mixer_dropout_0.2_head_l2_False,mlp_mixer,None,None,0.2,None,None,False
22,39b13dbb89bc41ee972420168cac5995,phase_3,val,36,11268,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.687695,0.825234,0.825234,0.0,...,trial_04_ast_mlp_256_learned_dropout_0.1_seed_0,0,ast_mlp_256_learned_dropout_0.1,ast,mlp_256,learned,0.1,None,None,None
24,3ae7ad287cc74928aceb5e17e767e804,phase_3,val,40,12520,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.690900,0.829081,0.829081,0.0,...,trial_22_ast_linear_learned_dropout_0.5_seed_2003,2003,ast_linear_learned_dropout_0.5,ast,linear,learned,0.5,None,None,None


## Phase 4

In [15]:
df4.config_str.unique()

array(['two_stage_detector_trial_01_ast_linear_interp_dropout_0.1_seed_0',
       'two_stage_detector_trial_01_mlp_mixer_dropout_0.0_head_l2_True_seed_0',
       'sampling_control_trial_01_convnext_sd_0.0_kernel_7_seed_0',
       'loss_reweighting_trial_01_convnext_sd_0.0_kernel_7_seed_0',
       'two_stage_detector_trial_01_convnext_sd_0.0_kernel_7_seed_0',
       'sampling_control_trial_01_ast_linear_interp_dropout_0.1_seed_0',
       'loss_reweighting_trial_01_ast_linear_interp_dropout_0.1_seed_0',
       'loss_reweighting_trial_01_mlp_mixer_dropout_0.0_head_l2_True_seed_0',
       'flat_multiclass_trial_01_ast_linear_interp_dropout_0.1_seed_0',
       'flat_multiclass_trial_01_convnext_sd_0.0_kernel_7_seed_0',
       'flat_multiclass_trial_01_mlp_mixer_dropout_0.0_head_l2_True_seed_0',
       'sampling_control_trial_01_mlp_mixer_dropout_0.0_head_l2_True_seed_0'],
      dtype=object)

In [16]:
ALLOWED_EVALUATION_STRATEGIES

{'flat_multiclass',
 'loss_reweighting',
 'sampling_control',
 'shared_two_head',
 'two_stage_detector'}

In [17]:
df4["eval_strat"] = df4["config_str"].apply(
    lambda s: next((strat for strat in ALLOWED_EVALUATION_STRATEGIES if f"{strat}" in s), None)
)

df4["model_family"] = df4["config_str"].apply(
    lambda s: next((family for family in ALLOWED_MODEL_FAMILIES if f"{family}" in s), None)
)

df4["ast_head"] = df4["config_str"].apply(
    lambda s: next((head for head in AST_HEAD if f"{head}" in s), None)
)

df4["ast_embedding"] = df4["config_str"].apply(
    lambda s: next((emb for emb in AST_EMB if f"{emb}" in s), None)
)

df4["dropout"] = df4["config_str"].apply(
    lambda s: (".".join(re.search(r"dropout_(\d+).(\d+)", s).groups()) if "dropout_" in s else None)
)

df4["kernel_size"] = df4["config_str"].apply(
    lambda s: re.search(r"kernel_(\d+)", s).group(1) if "kernel_" in s else None
)

df4["sd"] = df4["config_str"].apply(
    lambda s: ".".join(re.search(r"sd_(\d+).(\d+)", s).groups()) if "sd_" in s else None
)

df4["head_l2"] = df4["config_str"].apply(
    lambda s: True if "head_l2_True" in s else False if "head_l2_False" in s else None
)

In [18]:
df4

,run_id,phase,metric_phase,epoch,step,checkpoint_path,macro_f1,command_macro_f1,core_command_macro_f1,unknown_f1,...,trial_seed,config_str,eval_strat,model_family,ast_head,ast_embedding,dropout,kernel_size,sd,head_l2
3,113b2e8c8a6e404890dea59f78c58804,phase_4,val,13,15535,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,0,two_stage_detector_trial_01_ast_linear_interp_...,two_stage_detector,ast,linear,interp,0.1,None,None,None
8,1e7c001bf663478fbd092a6b3dcb4f13,phase_4,val,11,13145,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,42,two_stage_detector_trial_01_mlp_mixer_dropout_...,two_stage_detector,mlp_mixer,None,None,0.0,None,None,True
9,2262c543024b4282ba8833a582d0ca2c,phase_4,val,67,80065,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.887161,0.878181,0.878181,0.912908,...,0,sampling_control_trial_01_convnext_sd_0.0_kern...,sampling_control,convnext,None,None,None,7,0.0,None
10,29a88ff4404440b79d09cd58fe294c35,phase_4,val,14,16730,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,2003,two_stage_detector_trial_01_ast_linear_interp_...,two_stage_detector,ast,linear,interp,0.1,None,None,None
18,3566c77f5f2f425fa1a534a827d96241,phase_4,val,41,48995,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,2003,loss_reweighting_trial_01_convnext_sd_0.0_kern...,loss_reweighting,convnext,None,None,None,7,0.0,None
20,38fd557f4b4849fc8a346f6f997fd7ac,phase_4,val,11,13145,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,2003,two_stage_detector_trial_01_convnext_sd_0.0_ke...,two_stage_detector,convnext,None,None,None,7,0.0,None
21,397d06e10d2a4d359b46e475bf1fa02e,phase_4,val,12,14340,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.008542,0.010250,0.010250,0.000000,...,42,two_stage_detector_trial_01_convnext_sd_0.0_ke...,two_stage_detector,convnext,None,None,None,7,0.0,None
23,3a1cde766ce24ef980333ec97e30c429,phase_4,val,14,16730,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.426804,0.467291,0.467291,0.124853,...,0,sampling_control_trial_01_ast_linear_interp_dr...,sampling_control,ast,linear,interp,0.1,None,None,None
28,3d53cb0c20904980964189c021465e01,phase_4,val,15,17925,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.441933,0.452791,0.452791,0.041339,...,2003,sampling_control_trial_01_ast_linear_interp_dr...,sampling_control,ast,linear,interp,0.1,None,None,None
33,475cf352e4384a3ca748b09a311df5c2,phase_4,val,45,53775,/var/folders/8l/l8x5k0qj70d2ttx81bkn85n40000gn...,0.733768,0.712809,0.712809,0.713264,...,2003,loss_reweighting_trial_01_ast_linear_interp_dr...,loss_reweighting,ast,linear,interp,0.1,None,None,None


# Save

In [19]:
df1.drop(columns=["checkpoint_path", "trial_config", "config_str"], inplace=True)
df2.drop(columns=["checkpoint_path", "trial_config", "config_str"], inplace=True)
df3.drop(columns=["checkpoint_path", "trial_config", "config_str"], inplace=True)
df4.drop(columns=["checkpoint_path", "trial_config", "config_str"], inplace=True)

In [20]:
df1.to_json("analysis_data/phase_1.json", orient="records", lines=True)
df2.to_json("analysis_data/phase_2.json", orient="records", lines=True)
df3.to_json("analysis_data/phase_3.json", orient="records", lines=True)
df4.to_json("analysis_data/phase_4.json", orient="records", lines=True)

df_per_class.to_json("analysis_data/per_class_metrics.json", orient="records", lines=True)

In [22]:
df4.columns

Index(['run_id', 'phase', 'metric_phase', 'epoch', 'step', 'macro_f1',
       'command_macro_f1', 'core_command_macro_f1', 'unknown_f1', 'silence_f1',
       'macro_f1_nc', 'unknown_to_command_leakage',
       'silence_false_trigger_rate', 'trial_seed', 'eval_strat',
       'model_family', 'ast_head', 'ast_embedding', 'dropout', 'kernel_size',
       'sd', 'head_l2'],
      dtype='object')